# Notebook 04 — GitHub Workflow

**Module:** 00 — Orientation
**Notebook:** 04 of 13
**Prerequisites:** Notebooks 01–03
**Time estimate:** 60–90 minutes

Git is the local tool; GitHub is the remote platform. This notebook covers
the GitHub-specific features that matter for this program: repository visibility,
GitHub Actions CI, the `gh` CLI, and the "repository as portfolio artifact" mindset.

---
## Step 1 — Motivation

Your GitHub repository is not just storage. For Track A, it is the artifact a
hiring committee will click on before they meet you. For Track B, it is the
"demonstrated research output" that a PhD supervisor evaluates before deciding
whether to interview you.

This means the repository must be:
- **Public** — a private repo is invisible to both audiences
- **Readable at first glance** — the README must explain what this is and why it matters
- **Active** — a repo with 3 commits from 8 months ago signals an abandoned project
- **Green** — a passing CI badge is a visible, one-glance signal of code quality

---
## Step 2 — Intuition

Think of GitHub as a **lab website** for your computational work.
The README is the lab's homepage — it should answer "what do you do here?"
in under 5 minutes for a stranger. The commit history is the lab notebook.
The Actions CI is the automated quality control. The `portfolio/` folder is the
publication list.

The `gh` CLI is the command-line interface to GitHub itself — it lets you create
issues, view PR status, and check CI results without leaving the terminal.
This matters because staying in the terminal keeps you in flow.

---
## Step 3 — Biological Background

*Not applicable.*

---
## Step 4 — Mathematical Explanation

*Not applicable.*

---
## Step 5 — Computational Explanation

**GitHub Actions: continuous integration in 4 lines of YAML**

A CI workflow is a YAML file in `.github/workflows/`. When you push to the
repository, GitHub runs the workflow on a fresh virtual machine.
For this program, the CI workflow does three things:
1. Installs the Python environment from `requirements.txt`
2. Runs `pytest` on `utilities/compbio_utils/tests/`
3. Runs `ruff check` for linting

If any step fails, the push is flagged. This catches regressions the moment
they are introduced — not weeks later when you've forgotten what changed.

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — Check if gh CLI is installed and authenticated
import subprocess

def run(cmd: list[str]) -> tuple[int, str, str]:
    """Run a shell command; return (returncode, stdout, stderr)."""
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, r.stdout.strip(), r.stderr.strip()

code, out, err = run(["gh", "--version"])
if code == 0:
    print(f"gh CLI found:\n{out}")
else:
    print("gh CLI not found. Install from https://cli.github.com/")
    print("Then authenticate: gh auth login")

In [ ]:
# Cell 6.2 — View current repository information via gh
code, out, err = run(["gh", "repo", "view", "--json", "name,visibility,url,description"])
if code == 0:
    import json
    info = json.loads(out)
    print(f"Repository: {info.get('name', 'unknown')}")
    print(f"Visibility: {info.get('visibility', 'unknown')}")
    print(f"URL:        {info.get('url', 'unknown')}")
    print(f"Description:{info.get('description', '(none)')}")
else:
    print("Not yet pushed to GitHub, or gh not authenticated.")
    print("Run: gh auth login")

In [ ]:
# Cell 6.3 — Create a minimal GitHub Actions CI workflow
import pathlib

repo_root = pathlib.Path(__file__).resolve().parents[3]
workflow_dir = repo_root / ".github" / "workflows"
workflow_file = workflow_dir / "ci.yml"

ci_yaml = """\
name: CI

on:
  push:
    branches: ["main", "module-*"]
  pull_request:
    branches: ["main"]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: "pip"

      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install -e utilities/compbio_utils

      - name: Run pytest
        run: pytest utilities/compbio_utils/tests/ -v

      - name: Run ruff (lint)
        run: ruff check utilities/compbio_utils/
"""

workflow_dir.mkdir(parents=True, exist_ok=True)
if not workflow_file.exists():
    workflow_file.write_text(ci_yaml, encoding="utf-8")
    print(f"Created: {workflow_file.relative_to(repo_root)}")
else:
    print(f"Already exists: {workflow_file.relative_to(repo_root)}")

print("\nAfter pushing, CI will run automatically.")
print("Check status: gh run list")

In [ ]:
# Cell 6.4 — The elevator pitch as a one-liner (Track A + Track B)
# A PhD application or job application may ask for a one-sentence repository description.
# Write yours below.

elevator_pitch = """
This repository is a 12-month computational biology research accelerator —
a depth-tiered, reproducible, portfolio-driven program to become competitive
for Indian research positions (Track A, Month 6) and European PhD programmes
in computational biology / bioinformatics / systems biology (Track B, Month 12).
"""

print("Repository elevator pitch (edit this for your own voice):")
print(elevator_pitch)

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — GitHub repository anatomy
anatomy = """
  GitHub repository anatomy (relevant to this program):

  github.com/Vinoth-ai-20/computational-biology
  │
  ├── README.md ──────────────── Rendered on the repo homepage
  │                              This is what a PhD supervisor reads FIRST
  │
  ├── ○──○──○ Commit history ─── Every push visible here
  │                              A flat, inactive history is a red flag
  │
  ├── Actions ────────────────── CI runs: green badge = tests passing
  │   └── ci.yml (created above)
  │
  ├── portfolio/ ──────────────── Link this in your CV/SOP
  │
  └── Insights ────────────────── Contribution graph (visible on your profile page)
                                  Aim for a consistent streak, not a spike
"""
print(anatomy)

---
## Step 8 — Exercises

**Exercise 1:**
Make sure the repository is public: `gh repo edit --visibility public --yes`.
Confirm by visiting the URL from Cell 6.2 in a browser without being logged in.

**Exercise 2:**
Push the CI workflow created in Cell 6.3 to GitHub.
Wait for the Actions run to complete (check with `gh run list`).
Is it green or red? If red, diagnose and fix the failure.

**Exercise 3:**
Write a one-sentence elevator pitch for this repository in your own words
(different from Cell 6.4's template) in `exercises/04_github_exercises.md`.
The constraint: it must make sense to a PhD supervisor who has never heard of
"depth-tiered curriculum" or "research accelerator."

---
## Quiz — Active Recall

1. What three things does the CI workflow above check on every push?
2. Why must the repository be public?
3. What is the difference between `git remote` and `gh repo`?
4. What does a green CI badge signal to someone reading your repository cold?
5. What does `gh run list` show you?

---
## Reflection

**Date completed:** ____________________

**Reflection:**

> *[Is the CI workflow passing? If not, what failed and what did you learn from
> diagnosing it? What would a PhD supervisor or hiring manager see if they clicked
> on this repository today — is that what you want them to see?]*

---
## References

- [GitHub CLI manual](https://cli.github.com/manual/)
- [GitHub Actions documentation](https://docs.github.com/en/actions)
- [GitHub Docs: Making a repository public](https://docs.github.com/en/repositories/managing-your-repositorys-settings-and-features/managing-repository-settings/setting-repository-visibility)

---
*Next notebook: `04_markdown_for_scientific_writing.ipynb`*